# EDA — Prédiction de la délinquance (Geldium × Tata iQ)

Ce notebook présente l'analyse exploratoire du jeu de données `Delinquency_prediction_dataset.xlsx`, étape par étape :

1. Examiner le jeu de données et identifier les informations clés
2. Traiter les données manquantes et les problèmes de qualité
3. Détecter les schémas et les facteurs de risque
4. Produire le rapport EDA

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

DATA = "Delinquency_prediction_dataset.xlsx"
MONTHS = [f"Month_{i}" for i in range(1, 7)]
TARGET = "Delinquent_Account"

df = pd.read_excel(DATA)
df.head()

,Customer_ID,Age,Income,Credit_Score,Credit_Utilization,Missed_Payments,Delinquent_Account,Loan_Balance,Debt_to_Income_Ratio,Employment_Status,Account_Tenure,Credit_Card_Type,Location,Month_1,Month_2,Month_3,Month_4,Month_5,Month_6
0,CUST0001,56,165580.0,398.0,0.390502,3,0,16310.0,0.317396,EMP,18,Student,Los Angeles,Late,Late,Missed,Late,Missed,Late
1,CUST0002,69,100999.0,493.0,0.312444,6,1,17401.0,0.196093,Self-employed,0,Standard,Phoenix,Missed,Missed,Late,Missed,On-time,On-time
2,CUST0003,46,188416.0,500.0,0.359930,0,0,13761.0,0.301655,Self-employed,1,Platinum,Chicago,Missed,Late,Late,On-time,Missed,Late
3,CUST0004,32,101672.0,413.0,0.371400,3,0,88778.0,0.264794,Unemployed,15,Platinum,Phoenix,Late,Missed,Late,Missed,Late,Late
4,CUST0005,60,38524.0,487.0,0.234716,2,0,13316.0,0.510583,Self-employed,11,Standard,Phoenix,Missed,On-time,Missed,Late,Late,Late


---
## Étape 1 — Examiner le jeu de données et identifier les informations clés

### Prompts GenAI utilisés
- « Résume les principaux schémas, les valeurs aberrantes et les valeurs manquantes de ce jeu de données. Mets en évidence ce qui pourrait poser problème pour la modélisation de la délinquance. »
- « Identifie les 3 variables les plus susceptibles de prédire la délinquance et justifie brièvement. »
- « Vérifie la cohérence entre `Missed_Payments` et l'historique mensuel `Month_1` à `Month_6`. »

### 1.1 Structure

In [2]:
print(f"{df.shape[0]} lignes x {df.shape[1]} colonnes")
df.dtypes

500 lignes x 19 colonnes


Customer_ID                 str
Age                       int64
Income                  float64
Credit_Score            float64
Credit_Utilization      float64
Missed_Payments           int64
Delinquent_Account        int64
Loan_Balance            float64
Debt_to_Income_Ratio    float64
Employment_Status           str
Account_Tenure            int64
Credit_Card_Type            str
Location                    str
Month_1                     str
Month_2                     str
Month_3                     str
Month_4                     str
Month_5                     str
Month_6                     str
dtype: object

In [3]:
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Age,500.0,46.266,16.188,18.00,33.000,46.500,59.250,74.000
Income,461.0,108379.894,53662.724,15404.00,62295.000,107658.000,155734.000,199943.000
Credit_Score,498.0,577.717,168.881,301.00,418.250,586.000,727.250,847.000
Credit_Utilization,500.0,0.491,0.197,0.05,0.356,0.486,0.634,1.026
Missed_Payments,500.0,2.968,1.947,0.00,1.000,3.000,5.000,6.000
Delinquent_Account,500.0,0.160,0.367,0.00,0.000,0.000,0.000,1.000
Loan_Balance,471.0,48654.429,29395.537,612.00,23716.500,45776.000,75546.500,99620.000
Debt_to_Income_Ratio,500.0,0.299,0.095,0.10,0.234,0.302,0.363,0.553
Account_Tenure,500.0,9.740,5.923,0.00,5.000,10.000,15.000,19.000


### 1.2 Valeurs manquantes et doublons

In [4]:
na = df.isna().sum()
na = na[na > 0]
display(pd.DataFrame({"n": na, "%": (na / len(df) * 100).round(1)}))
print("Income ET Loan_Balance manquants :", (df.Income.isna() & df.Loan_Balance.isna()).sum())
print("Clients sans Credit_Score :", df.loc[df.Credit_Score.isna(), "Customer_ID"].tolist())
print("Customer_ID dupliqués :", df.Customer_ID.duplicated().sum())
print("Lignes dupliquées (hors ID) :", df.drop(columns="Customer_ID").duplicated().sum())

,n,%
Income,39,7.8
Credit_Score,2,0.4
Loan_Balance,29,5.8


Income ET Loan_Balance manquants : 4
Clients sans Credit_Score : ['CUST0116', 'CUST0379']
Customer_ID dupliqués : 0
Lignes dupliquées (hors ID) : 0


### 1.3 Variables catégorielles

In [5]:
for col in ["Employment_Status", "Credit_Card_Type", "Location", "Month_1"]:
    print(f"{col}: {df[col].value_counts().to_dict()}")

Employment_Status: {'Unemployed': 93, 'retired': 87, 'Employed': 82, 'EMP': 81, 'Self-employed': 80, 'employed': 77}
Credit_Card_Type: {'Gold': 118, 'Student': 112, 'Business': 108, 'Standard': 86, 'Platinum': 76}
Location: {'Los Angeles': 107, 'Phoenix': 103, 'Chicago': 103, 'Houston': 95, 'New York': 92}
Month_1: {'On-time': 177, 'Missed': 164, 'Late': 159}


### 1.4 Anomalies

In [6]:
print("Credit_Utilization > 1 :", (df.Credit_Utilization > 1).sum(),
      df.loc[df.Credit_Utilization > 1, "Credit_Utilization"].round(3).tolist())
print("Account_Tenure = 0 :", (df.Account_Tenure == 0).sum())

missed_6m = (df[MONTHS] == "Missed").sum(axis=1)
print(f"Missed_Payments == nb de 'Missed' sur Month_1..6 : {(missed_6m == df.Missed_Payments).mean():.1%} des lignes")
print("Corrélation Missed_Payments vs nb 'Missed' mensuels :", round(df.Missed_Payments.corr(missed_6m), 3))

Credit_Utilization > 1 : 4 [1.026, 1.025, 1.002, 1.009]
Account_Tenure = 0 : 28
Missed_Payments == nb de 'Missed' sur Month_1..6 : 14.4% des lignes
Corrélation Missed_Payments vs nb 'Missed' mensuels : 0.007


### 1.5 Variable cible et premiers indicateurs de risque

In [7]:
print(df[TARGET].value_counts().to_dict(), f"-> taux de délinquance {df[TARGET].mean():.1%}")
df.select_dtypes("number").corr()[TARGET].drop(TARGET).sort_values().round(3)

{0: 420, 1: 80} -> taux de délinquance 16.0%


Account_Tenure         -0.040
Missed_Payments        -0.026
Loan_Balance           -0.004
Age                     0.023
Credit_Utilization      0.034
Debt_to_Income_Ratio    0.034
Credit_Score            0.035
Income                  0.045
Name: Delinquent_Account, dtype: float64

In [8]:
for col in ["Employment_Status", "Credit_Card_Type", "Location"]:
    display(df.groupby(col)[TARGET].agg(taux="mean", n="count").round(3).sort_values("taux", ascending=False))

,taux,n
Employment_Status,,
Unemployed,0.194,93
Employed,0.183,82
Self-employed,0.162,80
employed,0.156,77
EMP,0.148,81
retired,0.115,87


,taux,n
Credit_Card_Type,,
Business,0.213,108
Student,0.179,112
Gold,0.144,118
Standard,0.128,86
Platinum,0.118,76


,taux,n
Location,,
Los Angeles,0.196,107
Houston,0.168,95
Phoenix,0.165,103
Chicago,0.146,103
New York,0.120,92


In [9]:
df.groupby(TARGET)[["Credit_Score", "Credit_Utilization", "Debt_to_Income_Ratio",
                    "Missed_Payments", "Income", "Loan_Balance", "Account_Tenure"]].mean().T.round(3)

Delinquent_Account,0,1
Credit_Score,575.146,591.150
Credit_Utilization,0.489,0.507
Debt_to_Income_Ratio,0.297,0.306
Missed_Payments,2.990,2.850
Income,107306.943,113902.013
Loan_Balance,48708.756,48358.233
Account_Tenure,9.843,9.200


### 1.6 Constats de l'étape 1

**Données manquantes ou incohérentes**
- **Income** : 39 valeurs manquantes (7,8 %), la variable la plus touchée.
- **Loan_Balance** : 29 valeurs manquantes (5,8 %).
- **Credit_Score** : 2 valeurs manquantes (0,4 %), CUST0116 et CUST0379.
- 4 clients n'ont ni `Income` ni `Loan_Balance`.
- **Employment_Status** n'est pas normalisé : `Employed` (82), `employed` (77) et `EMP` (81) désignent la même catégorie. `retired` est en minuscules. Il y a donc 6 libellés pour 4 catégories réelles.
- **Missed_Payments est incohérent avec l'historique mensuel** : il ne correspond au nombre de mois « Missed » de `Month_1` à `Month_6` que pour 14,4 % des clients (corrélation ≈ 0,01).
- Aucun doublon, ni sur `Customer_ID` ni sur les lignes complètes.

**Anomalies clés**
- **Credit_Utilization > 1** pour 4 clients (jusqu'à 1,026). C'est possible en cas de dépassement de limite, mais à confirmer avec Geldium.
- **Account_Tenure = 0** pour 28 clients : ce sont des comptes de moins d'un an, avec très peu d'historique.
- **Credit_Score** varie de 301 à 847 (moyenne 578), sans valeur hors de l'échelle 300–850.
- **Les classes sont déséquilibrées** : 80 délinquants (16 %) contre 420 (84 %). Il faudra privilégier F1/AUC et pondérer les classes.
- **Les signaux bruts sont très faibles** : aucune variable numérique n'a une corrélation supérieure à |0,05| avec la cible.

**Premiers indicateurs du risque de défaillance**

| Variable | Constat | Pourquoi elle compte |
|---|---|---|
| **Credit_Card_Type** | 21,3 % pour *Business* et 17,9 % pour *Student*, contre 11,8 % pour *Platinum* | C'est l'écart entre groupes le plus net. |
| **Employment_Status** | 19,4 % chez les *Unemployed* contre 11,5 % chez les *retired* | L'instabilité des revenus est un facteur de risque classique. |
| **Historique de paiement** | Prédicteur métier attendu, mais incohérent entre `Missed_Payments` et `Month_1..6` | À fiabiliser en priorité. |
| **Credit_Utilization / DTI** | Légèrement plus élevés chez les délinquants (0,507 contre 0,489, et 0,306 contre 0,297) | La tendance va dans le sens attendu, mais l'effet est faible. |
| **Location** | 19,6 % à Los Angeles contre 12,0 % à New York | À surveiller, mais la variable présente un risque de biais géographique. |

**Les 3 variables les plus prometteuses :** (1) l'historique de paiement, à condition de le fiabiliser, (2) `Credit_Utilization`, (3) `Employment_Status` une fois normalisé.

**Résumé de la qualité des données**

Le jeu de données est globalement exploitable : 500 enregistrements uniques, sans doublon, et seules 3 colonnes ont des valeurs manquantes (`Income` 7,8 %, `Loan_Balance` 5,8 %, `Credit_Score` 0,4 %). Plusieurs problèmes de cohérence doivent cependant être corrigés avant la modélisation. Le plus visible est `Employment_Status`, avec 6 libellés pour 4 catégories. Le plus grave est le désaccord entre `Missed_Payments` et l'historique mensuel, qui touche la variable de risque a priori la plus importante. On note aussi quelques valeurs extrêmes à valider (utilisation > 100 %, ancienneté nulle) et un déséquilibre des classes (16 % de délinquants). Enfin, les relations univariées avec la cible sont très faibles, ce qui laisse penser que la délinquance dépend de combinaisons de variables plutôt que d'un facteur isolé.